## Thesis Chapter 3: Neural underpinnings of DD and brain-behavior questions

All additional plots & stats that are not already in main DD paper
e.g. correlations

### nPRF model stuff

- figure on null results of group comparison in main paper
-

In [75]:
import numpy as np
import pandas as pd
import os.path as op

BIDS_ROOT = '/Users/mrenke/data/ds-dnumrisk'
group_df = pd.read_csv(op.join(BIDS_ROOT, 'group_assignment.csv')).set_index('subject')

PHENOTYPE_DIR = BIDS_ROOT + '/derivatives/phenotype'

from numrisk.behavior_general.measures_registry import _load_magjudge_bauer, _load_magjudge_probit, _load_decode_r
magjudge_bauer_pcm = _load_magjudge_bauer(variant = 'v4_choice') # , suffix_columns=False
magjudge_bauer_pcm_RDM = _load_magjudge_bauer(variant = 'v4_rdm')
magjudge_probit = _load_magjudge_probit()

nPRF_r = _load_decode_r()

df_comb = nPRF_r.join(magjudge_bauer_pcm_RDM, how='inner').join(magjudge_bauer_pcm, how='inner').join(magjudge_probit, how='inner')


In [28]:

import pingouin

for behav_var in ['gamma_magjudge','perceptual_noise_sd_v4_choice','perceptual_noise_sd_v4_rdm']: #  'perceptual_noise_sd_unbiased',
    cor = pingouin.corr(df_comb['neural_numsense_precision'], df_comb[behav_var], method='spearman')  # 'spearman' or 'shepard'
    r_, p_ = np.round(cor['r'].iloc[0], 2), np.round(cor['p-val'].iloc[0], 3)
    print(f'correlation neural_numsense_precision vs {behav_var}: r={r_}, p={p_}')



correlation neural_numsense_precision vs gamma_magjudge: r=0.14, p=0.264
correlation neural_numsense_precision vs perceptual_noise_sd_v4_choice: r=-0.13, p=0.302
correlation neural_numsense_precision vs perceptual_noise_sd_v4_rdm: r=-0.25, p=0.04


#### Connectivity

In [89]:
from numrisk.behavior_general.measures_registry import _load_npc_dispersion #_load_npc_pfm_net_area # only has DAN & visual2

npc_pfm_net_area = pd.read_csv(op.join(PHENOTYPE_DIR, 'netsPFM_NPC_allNets_atlas-caNets_DDnr_method-individual_area.csv'))
npc_pfm_net_area = npc_pfm_net_area.set_index(['subject', 'network']).unstack('network')
npc_pfm_net_area.columns = [f'{col[1]}_{col[0]}' for col in npc_pfm_net_area.columns]

npc_grad_dispersion = _load_npc_dispersion()

df_comb = df_comb.join(npc_pfm_net_area, how='inner').join(npc_grad_dispersion, how='inner')

In [90]:
print(f'Correlation NPC_dispersion vs: \n')
networks = npc_pfm_net_area.columns

for net_var in networks:
    cor = pingouin.corr(df_comb['NPC_dispersion'], df_comb[net_var], method='spearman')  # 'spearman' or 'shepard'
    r_, p_ = np.round(cor['r'].iloc[0], 2), np.round(cor['p-val'].iloc[0], 3)
    N_valid_datapoints = df_comb[['NPC_dispersion', net_var]].dropna().shape[0]
    print(f'{net_var}: r={r_}, p={p_}, N={N_valid_datapoints}')


Correlation NPC_dispersion vs: 

Auditory_size: r=-1.0, p=0.0, N=3
Cingulo-Opercular_size: r=-0.15, p=0.25, N=57
Default_size: r=0.18, p=0.192, N=56
Dorsal-attention_size: r=-0.33, p=0.009, N=64
Frontoparietal_size: r=0.41, p=0.026, N=30
Somatomotor_size: r=0.07, p=0.595, N=64
Visual2_size: r=0.5, p=0.0, N=64


In [91]:
npc_pfm_net_area.mean(axis=0).sort_values(ascending=False).to_frame(name='mean_area').join(npc_pfm_net_area.std(axis=0).sort_values(ascending=False).to_frame(name='SD_area'))

,mean_area,SD_area
Dorsal-attention_size,67.329,13.395
Somatomotor_size,13.152,7.478
Visual2_size,10.252,8.935
Cingulo-Opercular_size,2.649,2.989
Frontoparietal_size,1.875,3.463
Default_size,1.852,1.561
Auditory_size,1.250,0.734


##### Group comparison

In [103]:
## Group comparisons:
# Summary statistics table + uncorrected t-tests
from scipy import stats
npc_pfm_net_area = pd.read_csv(op.join(PHENOTYPE_DIR, 'netsPFM_NPC_allNets_atlas-caNets_DDnr_method-individual_area.csv'))
networks = npc_pfm_net_area['network'].unique()

rows = []
for net in networks:
    sub_data = npc_pfm_net_area[npc_pfm_net_area['network'] == net].set_index('subject').join(group_df, how='inner').reset_index()
    g0 = sub_data[sub_data['group'] == 0]['size'].values
    g1 = sub_data[sub_data['group'] == 1]['size'].values
    t, p = stats.ttest_ind(g0, g1) if (len(g0) > 1 and len(g1) > 1) else (np.nan, np.nan)
    rows.append({
        'network': net,
        'group0_mean': np.round(g0.mean(), 3),
        'group0_std':  np.round(g0.std(), 3),
        'group1_mean': np.round(g1.mean(), 3),
        'group1_std':  np.round(g1.std(), 3),
        't': np.round(t, 3), 'p_uncorrected':np.round(p, 4),
    })

stats_df = (pd.DataFrame(rows)
            .sort_values('p_uncorrected')
            .reset_index(drop=True))

# Bonferroni correction
n_tests = stats_df['p_uncorrected'].notna().sum()
stats_df['p_bonferroni'] = (stats_df['p_uncorrected'] * n_tests).clip(upper=1.0)

pd.set_option('display.float_format', '{:.3f}'.format)
display(stats_df)

,network,group0_mean,group0_std,group1_mean,group1_std,t,p_uncorrected,p_bonferroni
0,Visual2,6.452,6.144,13.936,9.521,-3.695,0.001,0.003
1,Dorsal-attention,71.858,12.719,62.938,12.325,2.827,0.006,0.038
2,Cingulo-Opercular,1.897,1.595,3.375,3.704,-1.910,0.061,0.368
3,Frontoparietal,1.153,1.281,2.396,4.261,-0.985,0.333,1.000
4,Somatomotor,12.907,7.802,13.388,7.022,-0.257,0.798,1.000
5,Default,1.826,1.596,1.879,1.495,-0.127,0.899,1.000
6,Auditory,1.462,0.636,0.827,0.000,NaN,NaN,NaN


## Gradient stuff:
newest in notebook: `parietal_patterns/gradients_noHalo/rep_groupDiffs_dParams.ipynb`